# 2c — Join, features, and the model table

**Inputs:**

| File | Contents |
|---|---|
| `data/curated/taxi_airport_hourly.parquet` | Table A — the spine, keyed on `(pickup_date, pickup_hour, airport)`. |
| `data/curated/flights_hourly.parquet` | Table B — scheduled and realised arrivals at JFK and LGA. |
| `data/curated/flights_hourly_ewr.parquet` | Newark arrivals on `(date, hour)`. |
| `data/curated/flight_column_roles.json` | The leakage contract written by 2b. |
| `data/landing/weather/weather_*.parquet` | Meteostat hourly observations [1] for JFK, LGA, EWR, and Central Park. |

**Outputs:**

| File | Contents |
|---|---|
| `data/curated/model_table.parquet` | One row per `(date, hour, airport)`, every feature, both targets, and the split. |
| `data/curated/model_table_roles.json` | Feature list, targets, and leak-if-unlagged columns, read by notebook 4. |
| `data/curated/shapes_model_table.json` | Headline shapes. |

## Flow

0. Load the three curated tables and rename Table A's key columns to `(date, hour)`.
1. Left-join Table B and the Newark table onto Table A, asserting the row count each time.
2. Clean the weather files — null audit, spine, bounded carry-forward, precipitation
   fallback, Central Park cross-check — and join the two airport series.
3. Derive the temporal and holiday features.
4. Build the lag features and compute the seasonal-naive benchmark.
5. Enforce the leakage rule on both the flight and taxi sides.
6. Materialise the train/test split as a column.
7. Validate, write the table, and write the roles manifest.

Table A is the spine and everything left-joins onto it. Nothing in this notebook is scaled,
centred, or standardised: that belongs inside the model fit in notebook 4, on the training
split alone.

Notebook 4 fits two models with different jobs — a Poisson GLM for `n_pickups` and a
gradient-boosted ensemble for `mean_total` — so this notebook builds a history for both
targets and declares each feature's group in the roles manifest.


In [1]:
"""Join the curated taxi, flight, and weather tables into the model table.

Table A (taxi airport-hours) is the spine. The flight, Newark, and weather
tables are left-joined onto it, temporal, holiday, and lag features are
derived, the leakage contract from notebook 2b is enforced, and the train/test
split is materialised as a column.

Writes `model_table.parquet` and the roles manifest that notebook 4 reads in
place of a hardcoded feature list.
"""

import json
import math
import sys
from pathlib import Path

from pyspark.sql import DataFrame, Window
from pyspark.sql import functions as F

sys.path.append(str(Path("..") / "scripts"))
from spark_utils import (  # noqa: E402
    ARRIVAL_AIRPORTS,
    create_spark_session,
    hour_spine,
)

# --- Paths -----------------------------------------------------------------
# Notebook is expected to run from `notebooks/`.
PROJECT_ROOT = Path("..").resolve()
CURATED_DIR = PROJECT_ROOT / "data" / "curated"
WEATHER_DIR = PROJECT_ROOT / "data" / "landing" / "weather"

# --- Study window ----------------------------------------------------------
# Identical to notebooks 2a and 2b. Inclusive of the start, exclusive of the end.
WINDOW_START = "2023-01-01"
WINDOW_END = "2024-07-01"
WINDOW_DAYS = 547
EXPECTED_ROWS = WINDOW_DAYS * 24 * len(ARRIVAL_AIRPORTS)  # 26,256
EXPECTED_SITE_HOURS = WINDOW_DAYS * 24                    # 13,128 per weather site

# --- Split -----------------------------------------------------------------
# Train on the calendar year 2023, test on January–June 2024. Forward in time.
TRAIN_END = "2024-01-01"      # exclusive
EXPECTED_TRAIN_ROWS = 365 * 24 * len(ARRIVAL_AIRPORTS)  # 17,520
EXPECTED_TEST_ROWS = 182 * 24 * len(ARRIVAL_AIRPORTS)   # 8,736

# --- Lags ------------------------------------------------------------------
LAG_WEEK_HOURS = 168   # same hour, one week earlier
LAG_DAY_HOURS = 24     # same hour, one day earlier
LAG_HOUR = 1           # the previous clock hour
TRAILING_DAYS = 7      # trailing mean of the same hour, over this many days

# --- Weather gap filling ---------------------------------------------------
# Two carry-forward limits: three hours for the smooth measures (temperature,
# humidity, pressure, wind) and one hour for the event-like ones
# (precipitation). Gaps beyond the limit are left null.
MAX_FILL_SMOOTH_HOURS = 3
MAX_FILL_EVENT_HOURS = 1

In [2]:
spark = create_spark_session(app_name="MAST30034 — join and features")
spark.sparkContext.setLogLevel("WARN")
spark.version

your 131072x1 screen size is bogus. expect trouble
26/08/20 01:44:09 WARN Utils: Your hostname, iphone resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/08/20 01:44:09 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/08/20 01:44:10 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


'3.5.1'

## Step 0 — Load and align

Table A's key columns are renamed from `(pickup_date, pickup_hour)` to `(date, hour)` here
and nowhere else, so that every join below is written on one key name. The three row counts
are checked before anything is joined.


In [3]:
table_a = (
    spark.read.parquet(str(CURATED_DIR / "taxi_airport_hourly.parquet"))
    .withColumnRenamed("pickup_date", "date")
    .withColumnRenamed("pickup_hour", "hour")
)
table_b = spark.read.parquet(str(CURATED_DIR / "flights_hourly.parquet"))
table_ewr = spark.read.parquet(str(CURATED_DIR / "flights_hourly_ewr.parquet"))

with open(CURATED_DIR / "flight_column_roles.json") as handle:
    flight_roles = json.load(handle)

# The spine defines the row count for everything that follows.
taxi_rows = table_a.count()
assert taxi_rows == EXPECTED_ROWS, (
    f"Table A has {taxi_rows:,} rows, expected {EXPECTED_ROWS:,}"
)
assert table_b.count() == EXPECTED_ROWS, "Table B is not on the same grid as Table A"
assert table_ewr.count() == EXPECTED_SITE_HOURS, (
    "Newark table is not a complete hourly grid"
)

print(f"Table A  {taxi_rows:,} rows  ×  {len(table_a.columns)} columns")
print(f"Table B  {EXPECTED_ROWS:,} rows  ×  {len(table_b.columns)} columns")
print(f"Newark   {EXPECTED_SITE_HOURS:,} rows  ×  {len(table_ewr.columns)} columns")

Table A  26,256 rows  ×  20 columns
Table B  26,256 rows  ×  14 columns
Newark   13,128 rows  ×  7 columns


## Step 1 — The joins, one assertion each

`join_checked` left-joins a frame onto the spine and asserts the row count afterwards. It
also checks the right-hand side for duplicate keys *before* joining, so a many-to-one join
is reported against the table that caused it rather than as an unexplained row count.

| Table | Key |
|---|---|
| `flights_hourly` | `(date, hour, airport)` |
| `flights_hourly_ewr` | `(date, hour)`, broadcast across both airports |
| `weather` | `(obs_date, obs_hour, site)` → `(date, hour, airport)` |

The Newark table is keyed on `(date, hour)` alone, so its columns are broadcast identically
to the JFK and LGA rows of the same hour.


In [4]:
def join_checked(
    left: DataFrame,
    right: DataFrame,
    on: list,
    expected_rows: int,
    label: str,
) -> DataFrame:
    """Left-join two frames and fail immediately if the row count changes.

    Checks the right-hand side for duplicate keys before joining, so a
    many-to-one join is reported against the table that caused it.

    Args:
        left: The spine. Its row count must be preserved.
        right: Frame to join on. Must be unique on ``on``.
        on: Join key columns, present in both frames.
        expected_rows: Row count the result must have.
        label: Name used in the printed confirmation and failure messages.

    Returns:
        The joined frame.

    Raises:
        AssertionError: If the right-hand side has duplicate keys, or if the
            join changed the row count.
    """
    right_rows = right.count()
    unique_keys = right.dropDuplicates(on).count()
    assert right_rows == unique_keys, (
        f"{label}: right-hand side has {right_rows - unique_keys:,} duplicate keys "
        f"on {on} and would multiply rows"
    )

    joined = left.join(right, on=on, how="left")
    actual = joined.count()
    assert actual == expected_rows, (
        f"{label}: {actual:,} rows after join, expected {expected_rows:,}"
    )

    print(f"{label:<28} {actual:,} rows  ×  {len(joined.columns)} columns")
    return joined


model_table = join_checked(
    table_a, table_b, ["date", "hour", "airport"], EXPECTED_ROWS, "+ flights (JFK, LGA)"
)
model_table = join_checked(
    model_table, table_ewr, ["date", "hour"], EXPECTED_ROWS, "+ flights (EWR)"
)

+ flights (JFK, LGA)         26,256 rows  ×  31 columns
+ flights (EWR)              26,256 rows  ×  36 columns


## Step 2 — Weather

The Meteostat files come straight from `download.py` into this notebook, so they are cleaned
here: rows retrieved per site against the 13,128 hours the window contains, then per-column
null rates, then the fill decisions.

Two failures are reported separately because they need different treatment. A shortfall in
*rows* is a missing hour, which the spine below restores as a null. A null *value* within a
row that exists is a missing measurement. `model=False` was passed to Meteostat at download,
so every null is a genuine gap rather than model output.


In [5]:
weather_paths = sorted(str(path) for path in WEATHER_DIR.glob("weather_*.parquet"))
assert weather_paths, (
    f"No weather files in {WEATHER_DIR}. Run: "
    "python scripts/download.py --skip-taxi --skip-bts"
)

weather_raw = (
    spark.read.parquet(*weather_paths)
    .withColumn("obs_date", F.col("obs_date").cast("date"))
    .withColumn("obs_hour", F.col("obs_hour").cast("int"))
)

WEATHER_MEASURES = ["temp", "rhum", "prcp", "snow", "wspd", "wpgt", "pres", "coco"]

# Rows retrieved per site, against the hours the window contains. A shortfall
# here is a missing row, not a missing value.
(
    weather_raw
    .groupBy("site")
    .agg(
        F.count("*").alias("rows"),
        F.min("obs_date").alias("first_date"),
        F.max("obs_date").alias("last_date"),
    )
    .withColumn("missing_hours", F.lit(EXPECTED_SITE_HOURS) - F.col("rows"))
    .orderBy("site")
    .show(truncate=False)
)

+----+-----+----------+----------+-------------+
|site|rows |first_date|last_date |missing_hours|
+----+-----+----------+----------+-------------+
|EWR |13126|2023-01-01|2024-06-30|2            |
|JFK |13126|2023-01-01|2024-06-30|2            |
|LGA |13126|2023-01-01|2024-06-30|2            |
|NYC |12669|2023-01-01|2024-06-30|459          |
+----+-----+----------+----------+-------------+



In [6]:
# Per-column null rates, per site. Printed before the fill decisions below,
# which depend on them.
null_rates = weather_raw.groupBy("site").agg(
    *[
        F.round(F.avg(F.col(column).isNull().cast("double")), 4).alias(column)
        for column in WEATHER_MEASURES
    ]
)
null_rates.orderBy("site").show(truncate=False)

+----+------+------+------+----+------+----+------+------+
|site|temp  |rhum  |prcp  |snow|wspd  |wpgt|pres  |coco  |
+----+------+------+------+----+------+----+------+------+
|EWR |0.0   |0.0   |0.0833|1.0 |0.0   |1.0 |6.0E-4|0.9735|
|JFK |0.0   |0.0   |0.1185|1.0 |2.0E-4|1.0 |0.0   |0.9597|
|LGA |0.0   |0.0   |0.0718|1.0 |1.0E-4|1.0 |2.0E-4|0.9666|
|NYC |1.0E-4|0.0017|1.0   |1.0 |0.2824|1.0 |3.0E-4|0.933 |
+----+------+------+------+----+------+----+------+------+



### What is kept, and how gaps are filled

Three columns are dropped on the null rates printed above: `snow` and `wpgt` are entirely
null at all four stations, and `coco` is null in roughly 96% of hours with every populated
value a fog code, so it cannot support a categorical weather feature. Precipitation is
therefore derived from the measured `prcp` gauge and the temperature instead.

The remaining measures are filled by carrying the last observation forward under two
policies and a strict lookback limit:

| Columns | Lookback |
|---|---|
| `temp`, `rhum`, `pres`, `wspd` | 3 hours |
| `prcp` | 1 hour |

The fill is **positional** — `rowsBetween(-n, 0)` over the site's rows — which equals an
n-hour lookback only if every hour is present exactly once. A complete spine per site is
therefore built first and the observations left-joined onto it, which also turns a missing
row into a null the fill can see.

Two flags leave this step: `weather_imputed` marks an hour whose temperature was filled,
and `weather_missing` marks one still null after filling.


In [7]:
SMOOTH_COLUMNS = ["temp", "rhum", "pres", "wspd"]
EVENT_COLUMNS = ["prcp"]
KEPT_COLUMNS = SMOOTH_COLUMNS + EVENT_COLUMNS

sites = weather_raw.select("site").distinct()
weather_spine = (
    hour_spine(spark, WINDOW_START, WINDOW_END, date_col="obs_date", hour_col="obs_hour")
    .crossJoin(sites)
)

weather = weather_spine.join(
    weather_raw.select("site", "obs_date", "obs_hour", *KEPT_COLUMNS),
    ["site", "obs_date", "obs_hour"],
    how="left",
)

site_hours = weather.groupBy("site").count().collect()
for row in site_hours:
    assert row["count"] == EXPECTED_SITE_HOURS, (
        f"{row['site']} has {row['count']:,} hours after the spine join"
    )
print(f"{len(site_hours)} sites × {EXPECTED_SITE_HOURS:,} hours")

# Positional carry-forward, bounded to the previous n rows — equal to the
# previous n hours over the completed spine.
def carry_forward(column: str, hours: int):
    """Last observation carried forward over a bounded window of rows.

    Args:
        column: Column to fill.
        hours: Maximum number of hours to look back. One row is one hour over
            the completed spine, so this is also a row count.

    Returns:
        A Column holding the filled value.
    """
    window = (
        Window.partitionBy("site")
        .orderBy("obs_date", "obs_hour")
        .rowsBetween(-hours, 0)
    )
    return F.last(F.col(column), ignorenulls=True).over(window)


weather = weather.withColumn("temp_observed", F.col("temp").isNotNull())
for column in SMOOTH_COLUMNS:
    weather = weather.withColumn(column, carry_forward(column, MAX_FILL_SMOOTH_HOURS))
for column in EVENT_COLUMNS:
    weather = weather.withColumn(column, carry_forward(column, MAX_FILL_EVENT_HOURS))

weather = (
    weather
    .withColumn(
        "weather_imputed", ~F.col("temp_observed") & F.col("temp").isNotNull()
    )
    .withColumn("weather_missing", F.col("temp").isNull())
    .drop("temp_observed")
)

weather.groupBy("site").agg(
    F.sum(F.col("weather_imputed").cast("int")).alias("hours_filled"),
    F.sum(F.col("weather_missing").cast("int")).alias("hours_still_missing"),
).orderBy("site").show()

4 sites × 13,128 hours
+----+------------+-------------------+
|site|hours_filled|hours_still_missing|
+----+------------+-------------------+
| EWR|           2|                  0|
| JFK|           2|                  0|
| LGA|           2|                  0|
| NYC|         299|                161|
+----+------------+-------------------+



### Precipitation: the neighbouring-station fallback

After the one-hour carry-forward, `prcp` is still missing for roughly one hour in twelve at
each airport. The three airport stations sit inside a twenty-kilometre triangle, so
precipitation is resolved in a fixed order of preference and the order used is recorded per
row in `prcp_source`:

1. the site's own gauge, where it reported;
2. the mean of whichever other airport stations reported that hour;
3. zero, flagged as assumed.

Central Park is excluded from the fallback: its `prcp` is null for the entire window.
`is_precip` and `is_freezing` are then derived from the resolved gauge and the temperature.


In [8]:
# Central Park never reports precipitation, so it is excluded.
PRECIP_FALLBACK_SITES = list(ARRIVAL_AIRPORTS) + ["EWR"]

# Citywide reference: the mean of whichever airport stations reported that
# hour. One row per (obs_date, obs_hour), asserted after the join.
citywide_precip = (
    weather
    .where(F.col("site").isin(PRECIP_FALLBACK_SITES))
    .groupBy("obs_date", "obs_hour")
    .agg(
        F.avg("prcp").alias("prcp_citywide"),
        F.count("prcp").alias("prcp_sites_reporting"),
    )
)

weather_rows = weather.count()
weather = weather.join(citywide_precip, ["obs_date", "obs_hour"], how="left")
assert weather.count() == weather_rows, (
    "The citywide precipitation join changed the row count"
)

FREEZING_C = 0.0

weather = (
    weather
    .withColumn(
        "prcp_source",
        F.when(F.col("prcp").isNotNull(), F.lit("observed"))
        .when(F.col("prcp_citywide").isNotNull(), F.lit("neighbouring station"))
        .otherwise(F.lit("assumed dry")),
    )
    .withColumn("prcp", F.coalesce("prcp", "prcp_citywide", F.lit(0.0)))
    .drop("prcp_citywide", "prcp_sites_reporting")
    .withColumnRenamed("temp", "temp_c")
    .withColumnRenamed("rhum", "rhum_pct")
    .withColumnRenamed("prcp", "prcp_mm")
    .withColumnRenamed("wspd", "wspd_kmh")
    .withColumnRenamed("pres", "pres_hpa")
    .withColumn("is_precip", F.col("prcp_mm") > 0)
    .withColumn(
        "is_freezing",
        F.col("is_precip") & (F.col("temp_c") <= FREEZING_C),
    )
    .withColumn("prcp_imputed", F.col("prcp_source") != "observed")
)

weather.groupBy("site", "prcp_source").count().orderBy("site", F.desc("count")).show()
weather.groupBy("site").agg(
    F.round(F.avg(F.col("is_precip").cast("double")), 4).alias("share_wet_hours"),
    F.round(F.avg(F.col("is_freezing").cast("double")), 4).alias("share_freezing_hours"),
    F.round(F.max("prcp_mm"), 1).alias("max_hourly_mm"),
).orderBy("site").show()

+----+--------------------+-----+
|site|         prcp_source|count|
+----+--------------------+-----+
| EWR|            observed|12540|
| EWR|neighbouring station|  498|
| EWR|         assumed dry|   90|
| JFK|            observed|12041|
| JFK|neighbouring station|  997|
| JFK|         assumed dry|   90|
| LGA|            observed|12688|
| LGA|neighbouring station|  350|
| LGA|         assumed dry|   90|
| NYC|neighbouring station|13038|
| NYC|         assumed dry|   90|
+----+--------------------+-----+

+----+---------------+--------------------+-------------+
|site|share_wet_hours|share_freezing_hours|max_hourly_mm|
+----+---------------+--------------------+-------------+
| EWR|         0.1086|              0.0026|         27.7|
| JFK|         0.1031|              0.0037|         31.2|
| LGA|         0.1068|              0.0027|         36.1|
| NYC|         0.1257|              0.0028|         17.4|
+----+---------------+--------------------+-------------+



### Central Park as a cross-check

Four sites were downloaded but only the two airport stations are joined to the model table.
Central Park is used here as an independent check that the airport readings are sane: the
hourly temperature spread across all reporting sites is summarised, and a large spread would
indicate a station reporting in the wrong units or the wrong timezone.


In [9]:
# Hourly temperature spread across sites: maximum minus minimum reading in the
# same hour, over the hours where every site reported.
spread = (
    weather
    .where(F.col("temp_c").isNotNull())
    .groupBy("obs_date", "obs_hour")
    .agg(
        F.max("temp_c").alias("max_temp"),
        F.min("temp_c").alias("min_temp"),
        F.count("*").alias("sites_reporting"),
    )
    .where(F.col("sites_reporting") == len(site_hours))
    .withColumn("spread_c", F.col("max_temp") - F.col("min_temp"))
)

spread.agg(
    F.count("*").alias("hours_compared"),
    F.round(F.avg("spread_c"), 2).alias("mean_spread_c"),
    F.round(F.percentile_approx("spread_c", 0.95), 2).alias("p95_spread_c"),
    F.round(F.max("spread_c"), 2).alias("max_spread_c"),
).show()

+--------------+-------------+------------+------------+
|hours_compared|mean_spread_c|p95_spread_c|max_spread_c|
+--------------+-------------+------------+------------+
|         12967|         2.37|         5.5|        13.4|
+--------------+-------------+------------+------------+



In [10]:
# Join the two airport series. `download.py` writes the site code and the
# airport code as the same string, which makes this a rename rather than a
# lookup; asserted, since a mismatch would fill the columns with nulls.
weather_airports = (
    weather
    .where(F.col("site").isin(list(ARRIVAL_AIRPORTS)))
    .withColumnRenamed("site", "airport")
    .withColumnRenamed("obs_date", "date")
    .withColumnRenamed("obs_hour", "hour")
)
assert (
    {row["airport"] for row in weather_airports.select("airport").distinct().collect()}
    == set(ARRIVAL_AIRPORTS)
), "Weather site codes do not match the airport codes used by Tables A and B"

model_table = join_checked(
    model_table,
    weather_airports,
    ["date", "hour", "airport"],
    EXPECTED_ROWS,
    "+ weather (JFK, LGA)",
).cache()

+ weather (JFK, LGA)         26,256 rows  ×  47 columns


26/08/20 01:44:41 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


## Step 3 — Temporal and holiday features

Hour of day, day of week, month, and year, plus weekend and holiday indicators.

`day_of_week` uses the ISO convention (1 = Monday) rather than Spark's `dayofweek`
(1 = Sunday), so the weekend is the contiguous range `{6, 7}`. It is declared categorical in
the roles manifest, not a quantity.

Cyclical `hour_sin` and `hour_cos` are added for the GLM, which otherwise cannot express
that hour 23 and hour 0 are adjacent without twenty-four dummies.

Holidays are hardcoded from the OPM federal holiday list [2] rather than taken from a
package dependency, so the feature cannot change between runs. Observed dates are listed
alongside actual ones where they differ, and the adjacent travel days around Thanksgiving
and Christmas are flagged with `kind = "adjacent"`.


In [11]:
# US federal holidays inside the study window [2], plus the adjacent travel
# days. `kind` separates the two.
HOLIDAYS = [
    # (date, name, kind)
    ("2023-01-01", "New Year's Day", "federal"),
    ("2023-01-02", "New Year's Day (observed)", "federal"),
    ("2023-01-16", "Martin Luther King Jr. Day", "federal"),
    ("2023-02-20", "Washington's Birthday", "federal"),
    ("2023-05-29", "Memorial Day", "federal"),
    ("2023-06-19", "Juneteenth", "federal"),
    ("2023-07-04", "Independence Day", "federal"),
    ("2023-09-04", "Labor Day", "federal"),
    ("2023-10-09", "Columbus Day", "federal"),
    ("2023-11-10", "Veterans Day (observed)", "federal"),
    ("2023-11-11", "Veterans Day", "federal"),
    ("2023-11-22", "Thanksgiving eve", "adjacent"),
    ("2023-11-23", "Thanksgiving Day", "federal"),
    ("2023-11-24", "Day after Thanksgiving", "adjacent"),
    ("2023-11-26", "Sunday after Thanksgiving", "adjacent"),
    ("2023-12-24", "Christmas Eve", "adjacent"),
    ("2023-12-25", "Christmas Day", "federal"),
    ("2023-12-26", "Day after Christmas", "adjacent"),
    ("2023-12-31", "New Year's Eve", "adjacent"),
    ("2024-01-01", "New Year's Day", "federal"),
    ("2024-01-15", "Martin Luther King Jr. Day", "federal"),
    ("2024-02-19", "Washington's Birthday", "federal"),
    ("2024-05-27", "Memorial Day", "federal"),
    ("2024-06-19", "Juneteenth", "federal"),
]

# One row per date, or the join below would multiply the table.
assert len({entry[0] for entry in HOLIDAYS}) == len(HOLIDAYS), "Duplicate holiday date"
assert all(WINDOW_START <= entry[0] < WINDOW_END for entry in HOLIDAYS), (
    "A holiday falls outside the study window"
)

holidays = (
    spark.createDataFrame(HOLIDAYS, ["holiday_date", "holiday_name", "holiday_kind"])
    .withColumn("date", F.to_date("holiday_date"))
    .drop("holiday_date")
)

model_table = join_checked(
    model_table, F.broadcast(holidays), ["date"], EXPECTED_ROWS, "+ holidays"
)

26/08/20 01:44:41 WARN HintErrorLogger: A join hint (strategy=broadcast) is specified but it is not part of a join relation.
26/08/20 01:44:43 WARN HintErrorLogger: A join hint (strategy=broadcast) is specified but it is not part of a join relation.


+ holidays                   26,256 rows  ×  49 columns


In [12]:
# ISO day of week: Spark's `dayofweek` is 1 = Sunday, so it is rotated to
# 1 = Monday.
iso_dow = ((F.dayofweek("date") + 5) % 7) + 1

model_table = (
    model_table
    .withColumn("day_of_week", iso_dow)
    .withColumn("day_name", F.date_format("date", "EEE"))
    .withColumn("month", F.month("date"))
    .withColumn("year", F.year("date"))
    .withColumn("is_weekend", F.col("day_of_week") >= 6)
    .withColumn("is_holiday", F.coalesce(F.col("holiday_kind") == "federal", F.lit(False)))
    .withColumn(
        "is_holiday_adjacent",
        F.coalesce(F.col("holiday_kind") == "adjacent", F.lit(False)),
    )
    # Hour as an angle, so 23:00 and 00:00 are adjacent.
    .withColumn("hour_sin", F.sin(2 * math.pi * F.col("hour") / 24))
    .withColumn("hour_cos", F.cos(2 * math.pi * F.col("hour") / 24))
    .drop("holiday_kind")
)

model_table.groupBy("is_holiday", "is_holiday_adjacent").agg(
    F.countDistinct("date").alias("days"),
    F.round(F.avg("n_pickups"), 1).alias("mean_pickups_per_hour"),
).orderBy(F.desc("is_holiday"), F.desc("is_holiday_adjacent")).show()

+----------+-------------------+----+---------------------+
|is_holiday|is_holiday_adjacent|days|mean_pickups_per_hour|
+----------+-------------------+----+---------------------+
|      true|              false|  18|                183.0|
|     false|               true|   6|                155.9|
|     false|              false| 523|                177.3|
+----------+-------------------+----+---------------------+



## Step 4 — Lags

`F.lag(column, 168)` steps back 168 *rows*, not 168 hours. The complete spine built in 2a
guarantees exactly 24 rows per airport per day with no gaps, which is what makes the
positional lag a wall-clock lag here.

Both targets get their own history:

| Feature | For | Meaning |
|---|---|---|
| `pickups_lag_168h` | Model 1 | Same hour, one week ago. Also the benchmark below. |
| `pickups_lag_24h` | Model 1 | Same hour, yesterday. |
| `pickups_lag_1h` | Model 1 | The previous clock hour. |
| `pickups_same_hour_7d` | Model 1 | Trailing mean of the same hour over the previous seven days. |
| `mean_total_lag_168h` | Model 2 | Same hour, one week ago. |
| `mean_total_same_hour_7d` | Model 2 | Trailing mean of the same hour over the previous seven days. |

The trailing means use a window partitioned by `(airport, hour)` and ordered by date, framed
`rowsBetween(-7, -1)`, so the frame can never see the current row.

The first week of 2023 has no weekly history — 336 rows — and is flagged `has_full_lags`
rather than deleted. The weekly lag is then verified against an independent date-arithmetic
self-join.


In [13]:
# Ordered over the complete grid, per airport, so a positional lag is a
# wall-clock lag.
by_hour = Window.partitionBy("airport").orderBy("date", "hour")
same_hour_history = (
    Window.partitionBy("airport", "hour")
    .orderBy("date")
    .rowsBetween(-TRAILING_DAYS, -1)
)

model_table = (
    model_table
    # --- Model 1: volume history -------------------------------------------
    .withColumn("pickups_lag_168h", F.lag("n_pickups", LAG_WEEK_HOURS).over(by_hour))
    .withColumn("pickups_lag_24h", F.lag("n_pickups", LAG_DAY_HOURS).over(by_hour))
    .withColumn("pickups_lag_1h", F.lag("n_pickups", LAG_HOUR).over(by_hour))
    .withColumn("pickups_same_hour_7d", F.avg("n_pickups").over(same_hour_history))
    # --- Model 2: value history --------------------------------------------
    # `F.avg` ignores nulls, so the trailing mean is taken over the days in
    # the window that had trips in this hour.
    .withColumn("mean_total_lag_168h", F.lag("mean_total", LAG_WEEK_HOURS).over(by_hour))
    .withColumn(
        "mean_total_same_hour_7d", F.avg("mean_total").over(same_hour_history)
    )
    .withColumn("has_full_lags", F.col("pickups_lag_168h").isNotNull())
)

incomplete = model_table.where(~F.col("has_full_lags")).count()
expected_incomplete = TRAILING_DAYS * 24 * len(ARRIVAL_AIRPORTS)
assert incomplete == expected_incomplete, (
    f"{incomplete:,} rows lack a weekly lag, expected {expected_incomplete:,}"
)
print(f"{incomplete:,} rows without a full week of history (the first week of 2023)")

# The weekly lag of a row must equal the pickup count 168 hours earlier,
# checked against a date-arithmetic join rather than trusted.
lookahead = model_table.alias("now").join(
    model_table.select(
        F.col("date").alias("then_date"),
        F.col("hour").alias("then_hour"),
        F.col("airport").alias("then_airport"),
        F.col("n_pickups").alias("then_pickups"),
    ).alias("then"),
    (F.col("now.airport") == F.col("then.then_airport"))
    & (F.date_sub(F.col("now.date"), 7) == F.col("then.then_date"))
    & (F.col("now.hour") == F.col("then.then_hour")),
    how="inner",
).where(F.col("now.pickups_lag_168h") != F.col("then.then_pickups")).count()
assert lookahead == 0, f"{lookahead:,} rows have a weekly lag that is not seven days back"
print("Weekly lag verified against a date-arithmetic join.")

336 rows without a full week of history (the first week of 2023)
Weekly lag verified against a date-arithmetic join.


### The benchmark

`pickups_lag_168h` on its own is a seasonal-naive forecast: this hour will be what it was
last week. Its error is computed here, on the test split and excluding the daylight-saving
rows, so that notebook 4 compares against a benchmark fixed before any model was fitted. The
same is done for `mean_total_lag_168h`.


In [14]:
def error_metrics(frame: DataFrame, actual: str, predicted: str) -> dict:
    """Root mean squared error and mean absolute error between two columns.

    Args:
        frame: Frame holding both columns.
        actual: Observed value column.
        predicted: Predicted value column.

    Returns:
        Mapping with ``n``, ``rmse``, and ``mae``. Rows where either column is
        null are excluded and reflected in ``n``.
    """
    error = F.col(actual) - F.col(predicted)
    row = (
        frame
        .where(F.col(actual).isNotNull() & F.col(predicted).isNotNull())
        .agg(
            F.count("*").alias("n"),
            F.sqrt(F.avg(F.pow(error, 2))).alias("rmse"),
            F.avg(F.abs(error)).alias("mae"),
        )
        .collect()[0]
    )
    return {"n": row["n"], "rmse": round(row["rmse"], 3), "mae": round(row["mae"], 3)}


test_rows = model_table.where(
    (F.col("date") >= F.lit(TRAIN_END).cast("date")) & (~F.col("dst_anomaly"))
)

baseline_pickups = error_metrics(test_rows, "n_pickups", "pickups_lag_168h")
baseline_value = error_metrics(test_rows, "mean_total", "mean_total_lag_168h")

print(f"Seasonal-naive benchmark on the test period ({TRAIN_END} onwards)")
print(f"  n_pickups   n={baseline_pickups['n']:,}  "
      f"RMSE {baseline_pickups['rmse']:.2f} pickups  "
      f"MAE {baseline_pickups['mae']:.2f}")
print(f"  mean_total  n={baseline_value['n']:,}  "
      f"RMSE ${baseline_value['rmse']:.2f}  "
      f"MAE ${baseline_value['mae']:.2f}")

Seasonal-naive benchmark on the test period (2024-01-01 onwards)
  n_pickups   n=8,734  RMSE 52.84 pickups  MAE 36.08
  mean_total  n=8,198  RMSE $10.13  MAE $5.18


## Step 5 — Enforce the leakage rule

Every column under `realised_lag_only` in `flight_column_roles.json` is an outcome of the
hour it describes. Each is lagged by one hour here, and the unlagged original is excluded
from the feature list in Step 7.

The taxi side has no roles file and needs the same treatment: every column of Table A except
the keys, the flags, and the two targets describes trips that have already happened in that
hour. All of them are declared leak-if-unlagged, and the five with predictive value are
carried as explicit one-hour lags.


In [15]:
# --- Flight side: enforce the contract written by 2b -----------------------
realised_columns = (
    flight_roles["realised_lag_only"] + flight_roles["ewr_realised_lag_only"]
)
missing = [column for column in realised_columns if column not in model_table.columns]
assert not missing, f"Columns declared by 2b are absent from the join: {missing}"

for column in realised_columns:
    model_table = model_table.withColumn(
        f"{column}_lag_1h", F.lag(column, LAG_HOUR).over(by_hour)
    )

# --- Taxi side: the same rule, declared here ------------------------------
TAXI_KEYS = ["date", "hour", "airport"]
TAXI_FLAGS = ["dst_anomaly"]
# Named separately from the leaking columns: the targets are same-hour
# outcomes by definition and are excluded from the feature list for that
# reason, not flagged as a hazard.
TARGETS = ["n_pickups", "mean_total"]
taxi_same_hour = [
    column for column in table_a.columns
    if column not in TAXI_KEYS + TAXI_FLAGS + TARGETS
]

# The five carried forward as lags. The rest stay in the table for notebook
# 3's descriptive work and are barred from the feature list by the manifest.
TAXI_LAGGED = [
    "mean_fare",
    "share_flat_fare",
    "mean_distance_mi",
    "mean_tip_ratio",
    "share_flex_fare",
]
assert set(TAXI_LAGGED) <= set(taxi_same_hour), "A lagged taxi column is not in Table A"

for column in TAXI_LAGGED:
    model_table = model_table.withColumn(
        f"{column}_lag_1h", F.lag(column, LAG_HOUR).over(by_hour)
    )

leak_if_unlagged = sorted(set(taxi_same_hour + realised_columns))
print(f"{len(leak_if_unlagged)} columns declared leak-if-unlagged")
print(f"{len(realised_columns) + len(TAXI_LAGGED)} one-hour lagged features built")

23 columns declared leak-if-unlagged
14 one-hour lagged features built


## Step 6 — Materialise the split

The split is written into the table as a column rather than applied as a date filter at each
model fit, so that both models provably see the same split and it can be verified by reading
one column.

Train is the calendar year 2023 (17,520 rows); test is January–June 2024 (8,736). The split
runs forward in time, as the specification requires.


In [16]:
model_table = model_table.withColumn(
    "split",
    F.when(F.col("date") < F.lit(TRAIN_END).cast("date"), F.lit("train"))
    .otherwise(F.lit("test")),
)

split_counts = {
    row["split"]: row["rows"]
    for row in model_table.groupBy("split").agg(F.count("*").alias("rows")).collect()
}
assert split_counts["train"] == EXPECTED_TRAIN_ROWS, split_counts
assert split_counts["test"] == EXPECTED_TEST_ROWS, split_counts
assert sum(split_counts.values()) == EXPECTED_ROWS, split_counts

print(f"train  {split_counts['train']:,} rows  (2023)")
print(f"test   {split_counts['test']:,} rows  ({TRAIN_END} to {WINDOW_END})")

train  17,520 rows  (2023)
test   8,736 rows  (2024-01-01 to 2024-07-01)


## Step 7 — Validate and write

Seven checks before the table is written.

Two are worth naming. The **decomposition identity** `n_pickups × mean_total =
sum_total_amount` is what the two-model design rests on, and is checked with a tolerance
since a distributed sum of doubles is not associative. The **null audit** requires every
column containing nulls to match one of the families documented in this notebook — the
first week's lags, the means of empty hours, the weather gaps, the delay columns where
nothing was observed, the neighbouring-hour columns at the window edges, and `holiday_name`
on ordinary days — and fails the run on anything else.


In [17]:
# 1. The grid is intact and each key appears exactly once.
assert model_table.count() == EXPECTED_ROWS
assert model_table.dropDuplicates(["date", "hour", "airport"]).count() == EXPECTED_ROWS

# 2. No pickup was lost or duplicated by the four joins.
joined_pickups = model_table.agg(F.sum("n_pickups")).collect()[0][0]
source_pickups = table_a.agg(F.sum("n_pickups")).collect()[0][0]
assert joined_pickups == source_pickups, (
    f"{joined_pickups:,} pickups after the joins, {source_pickups:,} in Table A"
)

# 3. The decomposition identity the two models rely on.
identity = model_table.where(F.col("n_pickups") > 0).agg(
    F.max(
        F.abs(F.col("n_pickups") * F.col("mean_total") - F.col("sum_total_amount"))
    ).alias("max_abs_error")
).collect()[0]["max_abs_error"]
assert identity < 0.01, f"n_pickups × mean_total does not reproduce revenue: {identity}"
print(f"Revenue decomposition holds to ${identity:.6f}")

# 4. Daylight saving is still flagged on exactly its six rows.
dst_rows = model_table.where(F.col("dst_anomaly")).count()
assert dst_rows == 6, f"{dst_rows} DST rows flagged, expected 6"

# 5. Model 2's support is smaller than Model 1's, by exactly the hours with
#    no pickups.
empty_hours = model_table.where(F.col("n_pickups") == 0).count()
undefined_value = model_table.where(F.col("mean_total").isNull()).count()
assert empty_hours == undefined_value, (
    "mean_total is null in a different set of hours from the empty ones"
)
print(f"{empty_hours:,} airport-hours with no pickups — Model 2 is undefined in these")

# 6. Shares still lie in [0, 1] after the joins and lags.
for column in ["share_flat_fare", "share_longhaul", "share_cancelled"]:
    out_of_range = model_table.where(
        F.col(column).isNotNull() & ~F.col(column).between(0, 1)
    ).count()
    assert out_of_range == 0, f"{column} out of range in {out_of_range:,} rows"

print("Structural checks passed.")

Revenue decomposition holds to $0.000000
1,180 airport-hours with no pickups — Model 2 is undefined in these
Structural checks passed.


In [18]:
# 7. The null audit. Every column with nulls must belong to a documented family.
null_counts = {
    column: count
    for column, count in model_table.select([
        F.sum(F.col(column).isNull().cast("int")).alias(column)
        for column in model_table.columns
    ]).collect()[0].asDict().items()
    if count
}

# Families, in the order they are tested. A column is explained by the first
# family it matches.
LAG_SUFFIXES = ("_lag_1h", "_lag_24h", "_lag_168h", "_same_hour_7d")
lagged_columns = [
    column for column in model_table.columns if column.endswith(LAG_SUFFIXES)
]
empty_hour_means = [
    "mean_fare", "median_fare", "mean_total", "mean_distance_mi",
    "mean_duration_min", "mean_passengers", "share_flat_fare", "share_flex_fare",
    "mean_tip_ratio",
]
weather_columns = [
    "temp_c", "rhum_pct", "prcp_mm", "wspd_kmh", "pres_hpa",
    "is_precip", "is_freezing",
]
nothing_scheduled_columns = [
    "share_longhaul", "share_cancelled", "ewr_share_cancelled",
]
delay_columns = [
    "mean_arr_delay_min", "share_delayed_15", "ewr_mean_arr_delay_min",
]
window_edge_columns = ["sched_arrivals_prev_hr", "sched_arrivals_next_hr"]

FAMILIES = [
    ("first-week lags and one-hour lags", lagged_columns),
    ("statistic over an empty subset of the hour's trips", empty_hour_means),
    ("weather gaps beyond the fill limit", weather_columns),
    ("nothing scheduled to land in the hour", nothing_scheduled_columns),
    ("no delay observed in the hour", delay_columns),
    ("first and last hour of the window", window_edge_columns),
    ("ordinary (non-holiday) days", ["holiday_name"]),
]

unexplained = []
print(f"{'column':<30} {'nulls':>8}  family")
for column, count in sorted(null_counts.items(), key=lambda item: -item[1]):
    family = next(
        (name for name, members in FAMILIES if column in members), None
    )
    print(f"{column:<30} {count:>8,}  {family or '*** UNEXPLAINED ***'}")
    if family is None:
        unexplained.append(column)

assert not unexplained, f"Undocumented nulls in: {unexplained}"
print("\nNull audit passed.")

column                            nulls  family
holiday_name                     25,104  ordinary (non-holiday) days
mean_arr_delay_min_lag_1h         5,355  first-week lags and one-hour lags
share_delayed_15_lag_1h           5,355  first-week lags and one-hour lags
mean_arr_delay_min                5,353  no delay observed in the hour
share_delayed_15                  5,353  no delay observed in the hour
share_cancelled_lag_1h            5,344  first-week lags and one-hour lags
share_longhaul                    5,342  nothing scheduled to land in the hour
share_cancelled                   5,342  nothing scheduled to land in the hour
ewr_mean_arr_delay_min_lag_1h     2,294  first-week lags and one-hour lags
ewr_mean_arr_delay_min            2,292  no delay observed in the hour
ewr_share_cancelled_lag_1h        2,246  first-week lags and one-hour lags
ewr_share_cancelled               2,244  nothing scheduled to land in the hour
mean_tip_ratio_lag_1h             1,542  first-week lags a

### The roles manifest

`model_table_roles.json` is to notebook 4 what `flight_column_roles.json` was to this
notebook: the feature list as data, so the modelling notebook reads it instead of hardcoding
column names.

Features are grouped by kind because the two models take different subsets. `flags` are not
features — `dst_anomaly` and `has_full_lags` exist so notebook 4 can exclude those rows, and
`weather_imputed`, `weather_missing`, and `prcp_source` record how much of the weather was
observed.

Two assertions guard the manifest: every declared feature must exist in the table, and no
declared feature may be a leak-if-unlagged column.


In [19]:
FEATURES = {
    "temporal": ["hour", "day_of_week", "month", "is_weekend",
                 "is_holiday", "is_holiday_adjacent"],
    "cyclical": ["hour_sin", "hour_cos"],
    "categorical": ["airport", "day_of_week", "month"],
    "flight_schedule": flight_roles["features"],
    "flight_realised_lagged": [
        f"{column}_lag_1h" for column in flight_roles["realised_lag_only"]
    ],
    "ewr": flight_roles["ewr_features"] + [
        f"{column}_lag_1h" for column in flight_roles["ewr_realised_lag_only"]
    ],
    "weather": ["temp_c", "rhum_pct", "prcp_mm", "wspd_kmh", "pres_hpa",
                "is_precip", "is_freezing"],
    "taxi_lagged": [f"{column}_lag_1h" for column in TAXI_LAGGED],
    "lags_volume": ["pickups_lag_1h", "pickups_lag_24h", "pickups_lag_168h",
                    "pickups_same_hour_7d"],
    "lags_value": ["mean_total_lag_168h", "mean_total_same_hour_7d"],
}

ROLES = {
    "key": ["date", "hour", "airport"],
    "rows": EXPECTED_ROWS,
    "targets": {
        "model_1_volume": "n_pickups",
        "model_2_value": "mean_total",
        "product": "expected revenue per airport-hour; reconciles with sum_total_amount",
    },
    "features": FEATURES,
    "benchmark": {
        "volume": "pickups_lag_168h",
        "value": "mean_total_lag_168h",
        "test_rmse_pickups": baseline_pickups["rmse"],
        "test_mae_pickups": baseline_pickups["mae"],
        "test_rmse_mean_total": baseline_value["rmse"],
        "test_mae_mean_total": baseline_value["mae"],
    },
    "flags": ["dst_anomaly", "has_full_lags", "weather_imputed", "weather_missing",
              "prcp_imputed", "prcp_source"],
    "leak_if_unlagged": leak_if_unlagged,
    "split": {
        "column": "split",
        "train": f"{WINDOW_START} to {TRAIN_END} (exclusive)",
        "test": f"{TRAIN_END} to {WINDOW_END} (exclusive)",
        "train_rows": EXPECTED_TRAIN_ROWS,
        "test_rows": EXPECTED_TEST_ROWS,
    },
    "note": (
        "Columns under leak_if_unlagged describe trips or arrivals that have "
        "already happened in the hour they key. They may enter a model only at "
        "a lag of at least one hour. Nothing in this table is scaled or "
        "centred: standardisation belongs inside the model fit, on the "
        "training split alone. Rows where has_full_lags is false, and rows "
        "where dst_anomaly is true, should be excluded from the fit."
    ),
}

# Every declared feature must exist, and no feature may be a leaking column.
declared = [column for group in FEATURES.values() for column in group]
absent = [column for column in declared if column not in model_table.columns]
assert not absent, f"Declared features absent from the table: {absent}"

leaking = sorted(set(declared) & set(leak_if_unlagged))
assert not leaking, f"Unlagged same-hour outcomes in the feature list: {leaking}"

with open(CURATED_DIR / "model_table_roles.json", "w") as handle:
    json.dump(ROLES, handle, indent=2)

print(f"{len(set(declared))} distinct features declared across {len(FEATURES)} groups")
print("No same-hour outcome reaches the feature list.")

43 distinct features declared across 10 groups
No same-hour outcome reaches the feature list.


In [20]:
(
    model_table
    .orderBy("date", "hour", "airport")
    .coalesce(1)
    .write
    .mode("overwrite")
    .parquet(str(CURATED_DIR / "model_table.parquet"))
)

weather_quality = model_table.agg(
    F.sum(F.col("weather_imputed").cast("int")).alias("filled"),
    F.sum(F.col("weather_missing").cast("int")).alias("missing"),
    F.sum(F.col("prcp_imputed").cast("int")).alias("prcp_imputed"),
).collect()[0]

shapes = {
    "model_table_rows": EXPECTED_ROWS,
    "model_table_columns": len(model_table.columns),
    "features_declared": len(set(declared)),
    "train_rows": split_counts["train"],
    "test_rows": split_counts["test"],
    "rows_without_full_lags": incomplete,
    "zero_pickup_hours": empty_hours,
    "hours_model_2_undefined": undefined_value,
    "weather_hours_carried_forward": weather_quality["filled"],
    "weather_hours_unobserved": weather_quality["missing"],
    "precipitation_hours_not_observed_at_site": weather_quality["prcp_imputed"],
    "benchmark_test_rmse_pickups": baseline_pickups["rmse"],
    "benchmark_test_mae_pickups": baseline_pickups["mae"],
    "benchmark_test_rmse_mean_total": baseline_value["rmse"],
    "benchmark_test_mae_mean_total": baseline_value["mae"],
}
with open(CURATED_DIR / "shapes_model_table.json", "w") as handle:
    json.dump(shapes, handle, indent=2)

shapes

{'model_table_rows': 26256,
 'model_table_columns': 79,
 'features_declared': 43,
 'train_rows': 17520,
 'test_rows': 8736,
 'rows_without_full_lags': 336,
 'zero_pickup_hours': 1180,
 'hours_model_2_undefined': 1180,
 'weather_hours_carried_forward': 4,
 'weather_hours_unobserved': 0,
 'precipitation_hours_not_observed_at_site': 1527,
 'benchmark_test_rmse_pickups': 52.835,
 'benchmark_test_mae_pickups': 36.076,
 'benchmark_test_rmse_mean_total': 10.129,
 'benchmark_test_mae_mean_total': 5.179}

In [21]:
model_table.select(
    "date", "hour", "airport", "n_pickups", "mean_total", "sched_arrivals",
    "temp_c", "prcp_mm", "is_precip", "pickups_lag_168h", "split",
).orderBy("date", "hour", "airport").show(8, truncate=False)

+----------+----+-------+---------+-----------------+--------------+------+-------+---------+----------------+-----+
|date      |hour|airport|n_pickups|mean_total       |sched_arrivals|temp_c|prcp_mm|is_precip|pickups_lag_168h|split|
+----------+----+-------+---------+-----------------+--------------+------+-------+---------+----------------+-----+
|2023-01-01|0   |JFK    |253      |70.12221343873514|0             |9.4   |0.5    |true     |NULL            |train|
|2023-01-01|0   |LGA    |15       |54.32866666666667|0             |12.2  |0.5    |true     |NULL            |train|
|2023-01-01|1   |JFK    |112      |64.77830357142854|1             |8.9   |0.5    |true     |NULL            |train|
|2023-01-01|1   |LGA    |24       |53.76499999999999|0             |11.7  |0.3    |true     |NULL            |train|
|2023-01-01|2   |JFK    |14       |65.7792857142857 |0             |7.8   |0.0    |false    |NULL            |train|
|2023-01-01|2   |LGA    |0        |NULL             |0          

In [22]:
spark.stop()

## References

1. Meteostat. *Hourly weather observations* and *data format documentation*.
   <https://meteostat.net/> and <https://dev.meteostat.net/formats.html>
2. US Office of Personnel Management. *Federal Holidays*.
   <https://www.opm.gov/policy-data-oversight/pay-leave/federal-holidays/>
